# Day 16: Tool Calling

LLMs can generate text. But what if they could **do things**?

With tool calling, you define functions, and the LLM can request to call them.

## How It Works

1. You define a function (e.g., `get_weather`)
2. You describe the function to the LLM
3. The LLM decides when to call it
4. You execute the function and return the result
5. The LLM uses the result to respond

The LLM doesn't actually run the code — it tells YOU to run it.

## Setup

In [52]:
from google import genai
from google.genai import types
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')
API_KEY = os.environ["GEMINI_API_KEY"]
client = genai.Client(api_key=API_KEY)

## Step 1: Define a Function

A simple function that returns ticket prices.

In [53]:
# Our "database" of ticket prices
ticket_prices = {
    "london": "$799",
    "paris": "$899",
    "tokyo": "$1400",
    "berlin": "$499",
    "new york": "$599"
}

def get_ticket_price(destination_city: str) -> str:
    """Get the price of a ticket to a destination city."""
    print(f"🔧 Tool called: get_ticket_price({destination_city})")
    city = destination_city.lower()
    price = ticket_prices.get(city, "Unknown")
    return f"Ticket to {destination_city}: {price}"

In [54]:
# Test the function directly
get_ticket_price("Paris")

🔧 Tool called: get_ticket_price(Paris)


'Ticket to Paris: $899'

## Step 2: Describe the Tool to the LLM

The LLM needs a schema describing what the function does and what parameters it takes.

In [55]:
# Define the tool schema
get_ticket_price_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_ticket_price",
            description="Get the price of a return ticket to a destination city.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "destination_city": types.Schema(
                        type=types.Type.STRING,
                        description="The city the customer wants to travel to"
                    )
                },
                required=["destination_city"]
            )
        )
    ]
)

print("✅ Tool defined: get_ticket_price")

✅ Tool defined: get_ticket_price


## Step 3: Let the LLM Call the Tool

In [56]:
# Ask a question that requires the tool
user_message = "How much is a ticket to Tokyo?"

response = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=user_message,
    config=types.GenerateContentConfig(
        tools=[get_ticket_price_tool]
    )
)

print(f"❓ User: {user_message}")
print(f"\n🤖 LLM Response:")
print(response.candidates[0].content.parts)

❓ User: How much is a ticket to Tokyo?

🤖 LLM Response:
[Part(
  function_call=FunctionCall(
    args={
      'destination_city': 'Tokyo'
    },
    name='get_ticket_price'
  )
)]


## Step 4: Handle the Tool Call

The LLM returned a function call request. We execute it and send the result back.

In [ ]:
def handle_tool_call(response):
    """Execute the tool call and return the result."""
    part = response.candidates[0].content.parts[0]
    
    if part.function_call:
        func_call = part.function_call
        if func_call.name == "get_ticket_price":
            city = func_call.args["destination_city"]
            result = get_ticket_price(city)
            return result
    return None

# Execute the tool
tool_result = handle_tool_call(response)
print(f"\n📋 Tool Result: {tool_result}")

## Step 5: Complete the Conversation

Send the tool result back to the LLM for a final response.

In [ ]:
# Build the conversation with tool result
part = response.candidates[0].content.parts[0]

follow_up = client.models.generate_content(
    model="gemini-2.5-flash-lite",
    contents=[
        types.Content(role="user", parts=[types.Part.from_text(text=user_message)]),
        types.Content(role="model", parts=[part]),
        types.Content(role="user", parts=[
            types.Part.from_function_response(
                name="get_ticket_price",
                response={"result": tool_result}
            )
        ])
    ],
    config=types.GenerateContentConfig(
        tools=[get_ticket_price_tool]
    )
)

print(f"💬 Final Answer: {follow_up.text}")

## Complete Flow: One Function

In [ ]:
def chat_with_tools(message):
    """Chat function that handles tool calls."""
    
    # First call - might request a tool
    response = client.models.generate_content(
        model="gemini-2.5-flash-lite",
        contents=message,
        config=types.GenerateContentConfig(
            tools=[get_ticket_price_tool]
        )
    )
    
    part = response.candidates[0].content.parts[0]
    
    # Check if it's a tool call
    if part.function_call:
        # Execute the tool
        tool_result = handle_tool_call(response)
        
        # Send result back for final answer
        follow_up = client.models.generate_content(
            model="gemini-2.5-flash-lite",
            contents=[
                types.Content(role="user", parts=[types.Part.from_text(text=message)]),
                types.Content(role="model", parts=[part]),
                types.Content(role="user", parts=[
                    types.Part.from_function_response(
                        name="get_ticket_price",
                        response={"result": tool_result}
                    )
                ])
            ],
            config=types.GenerateContentConfig(
                tools=[get_ticket_price_tool]
            )
        )
        return follow_up.text
    
    # No tool call - return direct response
    return response.text

In [ ]:
# Test: Question that needs the tool
print(chat_with_tools("What's the price for a flight to Paris?"))

In [ ]:
# Test: Question that doesn't need the tool
print(chat_with_tools("What's your name?"))

## Key Takeaways

1. **Tools extend LLM capabilities** — from generating text to taking actions
2. **The LLM doesn't run code** — it requests tool calls, you execute them
3. **Schema is critical** — the LLM needs a clear description of what each tool does
4. **Two-step process** — LLM requests tool → you execute → LLM responds with result

